# Data Governance — Data Security (MinIO Policies)

Creates four IAM users with role-based permissions across pipeline zones:

| Role | Landing | Trusted | Exploitation |
|---|---|---|---|
| pipeline_admin | RW | RW | RW |
| data_engineer | RW | RW | R |
| data_scientist | — | R | RW |
| analyst | R | R | R |

Uses `mc admin` to attach (MinIO Client) via Docker.

Prerequisites: MinIO + Docker running.

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

## Constants & role definitions — 4 IAM users with per-bucket permissions

In [ ]:
import json
import subprocess
from io import BytesIO

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ROOT_USER = os.environ.get("MINIO_ACCESS_KEY", "admin")
MINIO_ROOT_PASSWORD = os.environ.get("MINIO_SECRET_KEY", "password")

LANDING_BUCKET = os.environ.get("LANDING_ZONE_BUCKET", "landing-zone")
TRUSTED_BUCKET = os.environ.get("TRUSTED_ZONE_BUCKET", "trusted-zone")
EXPLOITATION_BUCKET = os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone")

MINIO_CONTAINER = os.environ.get("MINIO_CONTAINER", "cymatics-minio")

ROLES = {
    "pipeline_admin": {
        "password": "admin-cymatics-2026",
        "description": "Full access to all zones (pipeline admin & testing)",
        "buckets": {LANDING_BUCKET: "readwrite", TRUSTED_BUCKET: "readwrite", EXPLOITATION_BUCKET: "readwrite"},
    },
    "data_engineer": {
        "password": "engineer-cymatics-2026",
        "description": "Manages ingestion and trusted-zone processing",
        "buckets": {LANDING_BUCKET: "readwrite", TRUSTED_BUCKET: "readwrite", EXPLOITATION_BUCKET: "readonly"},
    },
    "data_scientist": {
        "password": "scientist-cymatics-2026",
        "description": "Builds models and embeddings from exploitation data",
        "buckets": {TRUSTED_BUCKET: "readonly", EXPLOITATION_BUCKET: "readwrite"},
    },
    "analyst": {
        "password": "analyst-cymatics-2026",
        "description": "Consumes dashboards and KPIs (read-only across all zones)",
        "buckets": {LANDING_BUCKET: "readonly", TRUSTED_BUCKET: "readonly", EXPLOITATION_BUCKET: "readonly"},
    },
}

for role, cfg in ROLES.items():
    buckets_str = ", ".join(f"{b}={a}" for b, a in cfg["buckets"].items())
    print(f"  {role}: {buckets_str}")

## Policy builders — generate IAM policy JSON per role

In [ ]:
def _build_policy(role_name, bucket_permissions):
    # Build an IAM policy document for a role.
    statements = []
    for bucket, access in bucket_permissions.items():
        if access == "readonly":
            statements.append({
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:ListBucket", "s3:GetBucketLocation"],
                "Resource": [f"arn:aws:s3:::{bucket}", f"arn:aws:s3:::{bucket}/*"],
            })
        elif access == "readwrite":
            statements.append({
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject",
                           "s3:ListBucket", "s3:GetBucketLocation",
                           "s3:ListMultipartUploadParts", "s3:AbortMultipartUpload"],
                "Resource": [f"arn:aws:s3:::{bucket}", f"arn:aws:s3:::{bucket}/*"],
            })
    return {"Version": "2012-10-17", "Statement": statements}

# Preview one policy.
print(json.dumps(_build_policy("analyst", ROLES["analyst"]["buckets"]), indent=2))

## mc admin helpers — run MinIO Client commands via docker exec

In [ ]:
def _run_mc(args, check=True):
    cmd = ["docker", "exec", MINIO_CONTAINER, "mc"] + args
    return subprocess.run(cmd, capture_output=True, text=True, timeout=30, check=check)

def _mc_alias_set():
    _run_mc(["alias", "set", "local", "http://localhost:9000", MINIO_ROOT_USER, MINIO_ROOT_PASSWORD])

def _mc_create_user(username, password):
    _run_mc(["admin", "user", "add", "local", username, password], check=False)

def _mc_create_policy(policy_name, policy_doc):
    cmd = ["docker", "exec", "-i", MINIO_CONTAINER, "mc", "admin", "policy", "create", "local", policy_name, "/dev/stdin"]
    subprocess.run(cmd, input=json.dumps(policy_doc), capture_output=True, text=True, timeout=30, check=False)

def _mc_attach_policy(policy_name, username):
    _run_mc(["admin", "policy", "attach", "local", policy_name, "--user", username], check=False)

print("mc helpers loaded.")

## Apply policies — create users, generate IAM docs, attach to users

In [ ]:
print("Registering MinIO alias...")
_mc_alias_set()

results = []
for role_name, role_config in ROLES.items():
    policy_name = f"cymatics-{role_name}"
    print(f"\n  ── Role: {role_name} — {role_config['description']}")

    _mc_create_user(role_name, role_config["password"])
    print(f"     User '{role_name}': created/exists")

    policy_doc = _build_policy(role_name, role_config["buckets"])
    _mc_create_policy(policy_name, policy_doc)
    print(f"     Policy '{policy_name}': created")

    _mc_attach_policy(policy_name, role_name)
    print(f"     Policy attached to user")

    for bucket, access in role_config["buckets"].items():
        icon = "RW" if access == "readwrite" else "R "
        print(f"       [{icon}]  {bucket}")

    results.append({"role": role_name, "policy": policy_name, "permissions": role_config["buckets"]})

## Access control matrix

In [ ]:
print(f"\n{'=' * 62}")
print("  Access Control Matrix")
print(f"{'─' * 62}")
print(f"  {'Role':<18} {'Landing':<12} {'Trusted':<12} {'Exploitation':<12}")
print(f"  {'─' * 54}")
for r in results:
    perms = r["permissions"]
    cols = []
    for b in [LANDING_BUCKET, TRUSTED_BUCKET, EXPLOITATION_BUCKET]:
        access = perms.get(b, "—")
        cols.append("RW" if access == "readwrite" else ("R" if access == "readonly" else "—"))
    print(f"  {r['role']:<18} {cols[0]:<12} {cols[1]:<12} {cols[2]:<12}")
print(f"{'=' * 62}")

## Verify access — test actual read/write per role

In [ ]:
from minio import Minio, S3Error

print("Verifying access controls...\n")
all_passed = True

for role_name, role_config in ROLES.items():
    client = Minio(MINIO_ENDPOINT, access_key=role_name,
                   secret_key=role_config["password"], secure=False)
    role_ok = True

    for bucket in [LANDING_BUCKET, TRUSTED_BUCKET, EXPLOITATION_BUCKET]:
        expected = role_config["buckets"].get(bucket)
        can_read = False
        try:
            for _ in client.list_objects(bucket, prefix="", recursive=False): break
            can_read = True
        except Exception: pass

        can_write = False
        test_key = f"governance/.access_test_{role_name}"
        try:
            data = b"access_test"
            client.put_object(bucket, test_key, BytesIO(data), len(data))
            client.remove_object(bucket, test_key)
            can_write = True
        except Exception: pass

        if expected == "readwrite":
            passed = can_read and can_write
        elif expected == "readonly":
            passed = can_read and not can_write
        else:
            passed = not can_read and not can_write

        icon = "✓" if passed else "✗"
        exp_label = {"readwrite": "RW", "readonly": "R", None: "—"}.get(expected, expected)
        actual = ("R" if can_read else "") + ("W" if can_write else "") or "—"
        print(f"  {icon} {role_name:<18} {bucket:<22} expected={exp_label:<4} actual={actual:<4}")
        if not passed:
            role_ok = False
            all_passed = False

print(f"\n  {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")